In [1]:
# Colab only: clone the repo and move into the project root
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone repository if it hasn't been cloned yet
    if not os.path.exists('/content/convolutional_vae_quishing'):
        !git clone https://github.com/AnhJoe/convolutional_vae_quishing.git

    # Change working directory to the project root
    os.chdir('/content/convolutional_vae_quishing')

In [2]:
#| output: false
# Local only: find the project root (the folder containing src/) and add it to the import path
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\convolutional_vae_quishing


# Floors

A **floor** is a deliberately simple detector. It uses no learned model and no URL pattern, only a few trivial properties of a URL (for example how long it is). A floor is fitted on benign URLs only. Its score is how far a URL sits from the typical benign URL, so an unusual URL gets a high score (higher means more likely malicious, as for every score in the project).

Floors matter because the data holds shortcuts: malicious URLs are shorter than benign ones, benign URLs rarely have an IP address as host, and so on (`1_eda.ipynb` section 4). A model that only learns these shortcuts looks good without learning anything about maliciousness. **The VAE is only useful if it beats its floor.**

- **Floor A** (the headline floor) uses one property, the URL length. The score is the absolute z-score (how many standard deviations the length is from the benign average).
- **Floor B** (reported alongside) uses six properties: URL length, whether it has a scheme such as `https://`, whether the host is an IP address, the number of host labels, the path length and the share of digits. The score is the Mahalanobis distance (a distance that accounts for how the properties vary together). A tiny ridge is added so that an almost constant property, such as the IP-host flag among benign URLs, does not break the calculation.

This notebook checks the shared pipeline, fits both floors on the benign train URLs, scores every validation and test URL, and saves the scores. The code lives in `src/floors.py` and `src/runner.py`. Run `2_data_processing.ipynb` first.

In [3]:
# Imports and settings
import json

import pandas as pd

from src import checks
from src.floors import FLOORS, fit_floor, floor_summary
from src.prepare_data import load_items
from src.runner import run_floors
from src.splits import SPLIT_NAMES

PROC_DIR = root / "data" / "processed"
RESULTS_DIR = root / "results"

c:\Users\joetn\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1. Data

`load_items` reads the items table built by `2_data_processing.ipynb` (one row per kept URL, with its label, split and URL features). The data checks below confirm that the table is the one everyone on the team should have.

In [4]:
# Load the items table and its summary, then count the URLs per split and class
items = load_items(PROC_DIR)
summary = json.loads((PROC_DIR / "items_summary.json").read_text())
items.groupby(["split", "cls"]).size().unstack().loc[SPLIT_NAMES]

cls,benign,malicious
split,,
train,300969,403020
validation,64494,86361
test,64494,86362


**Checks on the data.** The next cell runs a set of checks on the items table and prints one row per check. Each row is `pass` or `FAIL`, and the cell stops with an error if any check fails, so a broken table cannot reach a model by accident. The checks confirm that the table matches `items_summary.json` (the row counts and the data version, which everyone on the team should see as the same value), that every split has both classes, that no registered domain sits in two splits of the same class, and that the URL columns are consistent.

In [5]:
# Run the data checks on the items table (stops with an error if any check fails)
checks.data_checks(items, summary).report()

check,result,detail
row count matches items_summary.json,pass,"1,005,700 rows"
data version matches items_summary.json,pass,08d6fa626da8. Everyone on the team should see the same value
item_id equals the row position,pass,
label matches the class folder,pass,
rows per split and class match items_summary.json,pass,"train: 300,969 benign, 403,020 malicious; validation: 64,494 benign, 86,361 malicious; test: 64,494 benign, 86,362 malicious"
every split has benign and malicious rows,pass,
no domain appears in two splits of the same class,pass,
malicious share of every split is within 1 point of the overall share,pass,"train 0.572, validation 0.572, test 0.572 (overall 0.572)"
"no empty URL, and url_len equals the URL length",pass,
truncated is true exactly for URLs longer than 256,pass,"5,742 truncated"


**Checks on the model input.** The next cell encodes 3,000 random URLs and checks the arrays a sequence model receives: shape and type, ids inside the vocabulary, padding only at the end, the right number of real ids per URL, that decoding gives the URL back (with `?` for unknown characters), that a character outside printable ASCII becomes the unknown token and that a URL longer than 256 characters is cut off. The floors do not read these arrays, but the VAE and the supervised reference do, so the check is run once here.

In [6]:
# Run the model input checks on a random sample of URLs
checks.url_input_checks(items).report()

check,result,detail
"array shape is (n, 256) and the type is int16",pass,"(3000, 256), int16"
ids are between 0 and 96 (vocabulary size 97),pass,"smallest 0, largest 96"
padding (id 0) only comes after the last character,pass,
"each row has min(URL length, 256) real ids",pass,
"decode_url(encode_urls(url)) gives back the URL (cut at the maximum length, unknown characters as ?)",pass,
a URL with a character outside printable ASCII gets the unknown token (id 1),pass,20 URLs tried
a URL longer than 256 is cut off at 256,pass,20 URLs tried
unknown token share,info,0.0022% of the real ids in this sample


**Checks on the evaluation.** The next cell sends made-up scores through the same function that evaluates every model (`save_and_evaluate`) in a temporary folder, so nothing is written to `results/`. Random scores must give an AUROC near 0.5, an AUPRC near the malicious share and about 1% and 5% of benign test URLs flagged. Perfect scores must give 1 everywhere, reversed scores must give an AUROC of 0, and a score file with a missing item, an unknown item or a NaN must be refused.

In [7]:
# Run the evaluation checks with made-up scores in a temporary folder
checks.evaluation_checks(items).report()

check,result,detail
random scores give AUROC near 0.5,pass,AUROC 0.5002
random scores give AUPRC near the malicious share,pass,"AUPRC 0.5726, malicious share 0.5725"
random scores flag about 1% and 5% of test benign items at the validation thresholds,pass,0.99% and 5.10%
"perfect scores give AUROC 1, AUPRC 1 and recall 1 with no false alarms",pass,"AUROC 1.000, AUPRC 1.000, recall at 1% FPR 1.000"
reversed scores give AUROC 0 (higher must mean more malicious),pass,AUROC 0.000
a validation file that misses one item is refused,pass,
a validation file with an item that is not in the split is refused,pass,
scores with a missing value (NaN) are refused,pass,


## 2. Fit and score the floors

`run_floors` fits each floor on the benign train URLs, scores every validation and test URL, saves the scores to `results/<floor>/<score>/seed0/` and returns the test metrics. A floor has no randomness, so it has one run (seed 0).

In [8]:
# Fit both floors on the benign train rows, score validation and test, and show the test metrics
metrics = run_floors(items, RESULTS_DIR)
metrics

floorA_url: fitted on 300,969 benign train rows, features ['url_len']
floorB_url: fitted on 300,969 benign train rows, features ['url_len', 'has_scheme', 'host_is_ip', 'n_host_labels', 'path_len', 'digit_ratio']


,model,score,seed,n_benign,n_malicious,prevalence,auroc,auprc,validation_auroc,recall_at_1pct_fpr,test_fpr_at_1pct,recall_at_5pct_fpr,test_fpr_at_5pct
0,floorA_url,abs_z,0,64494,86362,0.57248,0.586838,0.630830,0.587697,0.013698,0.009660,0.030858,0.050191
1,floorB_url,mahalanobis,0,64494,86362,0.57248,0.763982,0.804116,0.766330,0.096153,0.010187,0.282219,0.049400


**What "ordinary" means to the floors.** The table below lists the average and the spread of every floor property over the benign train URLs. A URL far from these averages gets a high floor score.

In [9]:
# Mean and standard deviation of every floor B property in the benign train rows (floor A uses the first one only)
benign_train = items[(items["split"] == "train") & (items["cls"] == "benign")]
floor_summary(fit_floor(benign_train, FLOORS["floorB_url"]["features"]))

,mean,std
url_len,49.176297,24.347937
has_scheme,0.069685,0.254616
host_is_ip,0.000013,0.003646
n_host_labels,2.484272,0.617922
path_len,29.874728,22.194857
digit_ratio,0.068631,0.087943


**How to read the metrics table.** Malicious is the positive class and every split is about 57% malicious.

- `auroc` is the chance that a random malicious URL scores above a random benign URL. 0.5 is a coin flip and 1 is perfect.
- `auprc` is the average precision. A random scorer gets the malicious share (`prevalence`, about 0.57), not 0.5.
- `recall_at_1pct_fpr` and `recall_at_5pct_fpr` are the share of malicious test URLs flagged when the alarm threshold lets 1% (or 5%) of the benign validation URLs through. `test_fpr_at_*` is the share of benign test URLs flagged by the same threshold and should be close to 1% (or 5%).
- `validation_auroc` is the AUROC on the validation split, used to choose between scores before the test set is looked at.

Floor B is stricter than floor A, so its AUROC is the one the VAE has to be compared with first when asking whether the VAE learned more than scheme, host and length. Confidence intervals and the paired comparison with the VAE are in `6_results.ipynb`.